# Pulsar candidate classification: full model comparison
**Supervised:** XGBoost, Random Forest, SVM (RBF)  |  **Unsupervised:** K-means (k=2), Isolation Forest

Everything is trained on `train.csv`. `val.csv` is used for tuning / early stopping / choosing each model's decision threshold. `test.csv` is used **once** for the final comparison.

Runtime: roughly 10 to 20 minutes on Colab (the SVM search is the slowest part).

In [ ]:
!pip -q install xgboost

In [ ]:
import os, time, json, shutil, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, fbeta_score, matthews_corrcoef, cohen_kappa_score,
                             confusion_matrix, roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve, silhouette_score)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
os.makedirs("figures", exist_ok=True)
def savefig(name):
    plt.savefig(f"figures/{name}.png", bbox_inches="tight", dpi=200)

# In Colab: upload train.csv, val.csv, test.csv when prompted (the files from split_data.py)
try:
    from google.colab import files
    if not all(os.path.exists(f) for f in ["train.csv", "val.csv", "test.csv"]):
        print("Upload train.csv, val.csv and test.csv")
        files.upload()
except ImportError:
    pass  # running locally

DATA_DIR = "data" if os.path.exists("data/train.csv") else "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
val   = pd.read_csv(f"{DATA_DIR}/val.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

FEATURES = [c for c in train.columns if c != "label"]
X_train, y_train = train[FEATURES].values, train["label"].astype(int).values
X_val,   y_val   = val[FEATURES].values,   val["label"].astype(int).values
X_test,  y_test  = test[FEATURES].values,  test["label"].astype(int).values

SEED = 42
MIN_PRECISION = 0.90   # rule used to pick EVERY model's threshold on validation data (lower it to favour recall)
COLORS = {"XGBoost": "#1f77b4", "Random Forest": "#2ca02c", "SVM (RBF)": "#ff7f0e",
          "Isolation Forest": "#9467bd", "K-means (k=2)": "#7f7f7f"}
SUPERVISED = ["XGBoost", "Random Forest", "SVM (RBF)"]
print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)
print("pulsar ratio train/val/test:", y_train.mean().round(4), y_val.mean().round(4), y_test.mean().round(4))

## 1. Quick look at the data
The big imbalance (about 9% pulsars) is why accuracy alone is misleading, so we also track recall, precision, F1/F2, MCC and PR-AUC.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
n_neg = int((np.r_[y_train, y_val, y_test] == 0).sum()); n_pos = int((np.r_[y_train, y_val, y_test] == 1).sum())
b = ax[0].bar(["not pulsar", "pulsar"], [n_neg, n_pos], color=["#9aa5b1", "#d62728"])
for r, v in zip(b, [n_neg, n_pos]):
    ax[0].text(r.get_x() + r.get_width() / 2, v, f"{v:,}", ha="center", va="bottom")
ax[0].set_title("Class balance (all data)")
sizes = {"train": len(y_train), "val": len(y_val), "test": len(y_test)}
pos = {"train": y_train.sum(), "val": y_val.sum(), "test": y_test.sum()}
x = np.arange(3)
ax[1].bar(x, [sizes[k] - pos[k] for k in sizes], label="not pulsar", color="#9aa5b1")
ax[1].bar(x, [pos[k] for k in sizes], bottom=[sizes[k] - pos[k] for k in sizes], label="pulsar", color="#d62728")
ax[1].set_xticks(x); ax[1].set_xticklabels(list(sizes)); ax[1].set_title("Split sizes"); ax[1].legend()
plt.tight_layout(); savefig("01_class_balance"); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(14, 5.5))
for a, f in zip(axes.ravel(), FEATURES):
    lo, hi = np.percentile(train[f], [0.5, 99.5])
    a.hist(train.loc[train.label == 0, f].clip(lo, hi), bins=40, alpha=.6, density=True, label="not pulsar", color="#9aa5b1")
    a.hist(train.loc[train.label == 1, f].clip(lo, hi), bins=40, alpha=.6, density=True, label="pulsar", color="#d62728")
    a.set_title(f, fontsize=10)
axes[0, 0].legend(fontsize=8)
plt.suptitle("Feature distributions by class (train)", y=1.02); plt.tight_layout()
savefig("02_feature_distributions"); plt.show()

## 2. Helpers
All models are scored the same way: each produces a continuous score (higher = more pulsar-like), a threshold is picked on validation data using the same rule, and everything is evaluated on the same test set.

In [ ]:
def pick_threshold(y_true, scores, min_precision=MIN_PRECISION):
    """Highest-recall threshold that keeps precision >= min_precision (fallback: best F1). Use on VALIDATION only."""
    prec, rec, thr = precision_recall_curve(y_true, scores)
    prec, rec = prec[:-1], rec[:-1]
    ok = prec >= min_precision
    if ok.any():
        i = np.argmax(np.where(ok, rec, -1))
    else:
        f1 = 2 * prec * rec / (prec + rec + 1e-12); i = np.argmax(f1)
    return float(thr[i])

def full_metrics(y, s, thr):
    pred = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"Accuracy": accuracy_score(y, pred),
            "Balanced Acc": balanced_accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall (TPR)": recall_score(y, pred),
            "Specificity (TNR)": tn / (tn + fp),
            "F1": f1_score(y, pred),
            "F2 (recall-weighted)": fbeta_score(y, pred, beta=2),
            "MCC": matthews_corrcoef(y, pred),
            "Cohen kappa": cohen_kappa_score(y, pred),
            "Miss rate (FNR)": fn / (fn + tp),
            "False alarm rate (FPR)": fp / (fp + tn),
            "ROC-AUC": roc_auc_score(y, s),
            "PR-AUC": average_precision_score(y, s),
            "False Negatives": int(fn), "False Positives": int(fp), "True Positives": int(tp), "True Negatives": int(tn)}

RES = {}   # name -> scores on train/val/test, thresholds, timings
def register(name, kind, f_score, fit_time, default_thr):
    t0 = time.time(); s_test = f_score(X_test); pred_time = time.time() - t0
    s_val = f_score(X_val)
    RES[name] = dict(kind=kind, train=f_score(X_train), val=s_val, test=s_test,
                     default_thr=default_thr, thr=pick_threshold(y_val, s_val),
                     fit_time=fit_time, pred_time=pred_time)
    r = RES[name]
    print(f"{name}: tuned threshold = {r['thr']:.3f} | fit {fit_time:.1f}s | test PR-AUC = {average_precision_score(y_test, s_test):.4f}")

## 3. Supervised model 1: XGBoost
`scale_pos_weight` = non-pulsars / pulsars handles the imbalance. Shallow trees (depth 3 to 5) limit overfitting, and early stopping on the validation set picks the number of trees.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import ParameterSampler
from scipy.stats import randint, uniform, loguniform

spw = (y_train == 0).sum() / (y_train == 1).sum()
space = {"max_depth": randint(3, 6), "learning_rate": loguniform(0.01, 0.3),
         "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.6, 0.4),
         "min_child_weight": randint(1, 10), "reg_lambda": loguniform(0.1, 10)}
N_ITER = 20
t0 = time.time(); best = {"score": -1}
for k, params in enumerate(ParameterSampler(space, n_iter=N_ITER, random_state=SEED)):
    m = XGBClassifier(n_estimators=1000, scale_pos_weight=spw, eval_metric="aucpr",
                      early_stopping_rounds=50, random_state=SEED, n_jobs=-1, **params)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    s = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
    if s > best["score"]:
        best = {"score": s, "params": params, "model": m}
xgb_fit_time = time.time() - t0
xgb = best["model"]
print("Best params:", {k: round(float(v), 4) for k, v in best["params"].items()}, "| trees:", xgb.best_iteration + 1)
register("XGBoost", "supervised", lambda X: xgb.predict_proba(X)[:, 1], xgb_fit_time, 0.5)

## 4. Supervised model 2: Random Forest
`class_weight="balanced_subsample"`, tuned with stratified 5-fold CV on the train set (scored by PR-AUC).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
t0 = time.time()
rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight="balanced_subsample", random_state=SEED, n_jobs=-1),
    {"n_estimators": randint(100, 500), "max_depth": randint(4, 25),
     "min_samples_leaf": randint(1, 10), "max_features": ["sqrt", "log2", 0.5]},
    n_iter=20, cv=cv, scoring="average_precision", random_state=SEED, n_jobs=-1)
rf_search.fit(X_train, y_train)
rf_fit_time = time.time() - t0
rf = rf_search.best_estimator_
print("Best params:", rf_search.best_params_, "| CV PR-AUC:", round(rf_search.best_score_, 4))
register("Random Forest", "supervised", lambda X: rf.predict_proba(X)[:, 1], rf_fit_time, 0.5)

## 5. Supervised model 3: SVM (RBF)
Features standardized inside a pipeline, `class_weight="balanced"`, C and gamma tuned by 5-fold CV. Scored with the decision function (default cut-off 0), which is much faster than `probability=True`.

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

t0 = time.time()
svm_search = RandomizedSearchCV(
    Pipeline([("scale", StandardScaler()), ("svm", SVC(kernel="rbf", class_weight="balanced", cache_size=1000))]),
    {"svm__C": loguniform(0.1, 100), "svm__gamma": loguniform(1e-3, 1)},
    n_iter=15, cv=cv, scoring="average_precision", random_state=SEED, n_jobs=-1)
svm_search.fit(X_train, y_train)
svm_fit_time = time.time() - t0
svm = svm_search.best_estimator_
print("Best params:", svm_search.best_params_, "| CV PR-AUC:", round(svm_search.best_score_, 4))
register("SVM (RBF)", "supervised", lambda X: svm.decision_function(X), svm_fit_time, 0.0)

## 6. Unsupervised models
These **never see labels while fitting**. They are fit on `train.csv` features only (standardized).

* **K-means (k=2):** finds two clusters. Labels are used only afterwards to *name* which cluster is "pulsar" (the one with the higher pulsar fraction in train). The score is "distance to the non-pulsar centroid minus distance to the pulsar centroid"; the default cut-off 0 is plain K-means assignment.
* **Isolation Forest:** treats pulsars as anomalies (they are rare). The default cut-off flags the top ~9% most anomalous points. Its score direction is checked on validation data.

Like the supervised models, their threshold is tuned on `val.csv`, so the comparison stays fair (but note this does use validation labels). If no threshold can reach the 90% precision target, the rule falls back to the best-F1 threshold, which is common for the unsupervised models.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

sc = StandardScaler().fit(X_train)
Xtr_s, Xva_s, Xte_s = sc.transform(X_train), sc.transform(X_val), sc.transform(X_test)
to_scaled = {id(X_train): Xtr_s, id(X_val): Xva_s, id(X_test): Xte_s}
def scaled(X): return to_scaled[id(X)]

# ---- K-means ----
t0 = time.time()
km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(Xtr_s)
km_time = time.time() - t0
cl = km.predict(Xtr_s)
pulsar_cluster = int(pd.Series(y_train).groupby(cl).mean().idxmax())
def km_score(X):
    d = km.transform(scaled(X)); return d[:, 1 - pulsar_cluster] - d[:, pulsar_cluster]
register("K-means (k=2)", "unsupervised", km_score, km_time, 0.0)

# ---- Isolation Forest ----
t0 = time.time()
iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Xtr_s)
iso_time = time.time() - t0
raw = lambda X: -iso.score_samples(scaled(X))          # higher = more anomalous
sign = 1 if roc_auc_score(y_val, raw(X_val)) >= 0.5 else -1
if sign == -1:
    print("Note: pulsars look LESS anomalous than the majority here, so the score direction was flipped (decided on validation).")
iso_score = lambda X: sign * raw(X)
iso_default = float(np.quantile(iso_score(X_train), 1 - y_train.mean()))   # flag top ~9%
register("Isolation Forest", "unsupervised", iso_score, iso_time, iso_default)

## 7. Metrics: all five models on the test set
Two views: with each model's **tuned threshold** (picked on validation to catch as many pulsars as possible while keeping precision >= 90%), and with its **default threshold**. Green = best in that column.

In [ ]:
HIGHER = ["Accuracy", "Balanced Acc", "Precision", "Recall (TPR)", "Specificity (TNR)", "F1",
          "F2 (recall-weighted)", "MCC", "Cohen kappa", "ROC-AUC", "PR-AUC", "True Positives", "True Negatives"]
LOWER = ["Miss rate (FNR)", "False alarm rate (FPR)", "False Negatives", "False Positives"]
INTCOLS = ["False Negatives", "False Positives", "True Positives", "True Negatives"]

def show_table(df, title):
    print("\n" + title)
    try:
        from IPython.display import display
        floatcols = [c for c in df.columns if c not in INTCOLS]
        sty = (df.style.format("{:.4f}", subset=floatcols).format("{:.0f}", subset=[c for c in INTCOLS if c in df.columns])
                 .highlight_max(subset=[c for c in HIGHER if c in df.columns], color="#c6efce")
                 .highlight_min(subset=[c for c in LOWER if c in df.columns], color="#c6efce"))
        display(sty)
    except Exception:
        print(df.round(4).to_string())

table_tuned   = pd.DataFrame({n: full_metrics(y_test, r["test"], r["thr"]) for n, r in RES.items()}).T
table_default = pd.DataFrame({n: full_metrics(y_test, r["test"], r["default_thr"]) for n, r in RES.items()}).T
for t in (table_tuned, table_default):
    for c in INTCOLS: t[c] = t[c].astype(int)
    for c in t.columns:
        if c not in INTCOLS: t[c] = t[c].astype(float)
show_table(table_tuned, "TEST SET: tuned threshold")
show_table(table_default, "TEST SET: default threshold (0.5 for XGB/RF, 0 for SVM/K-means, top-9% for Isolation Forest)")

extra = pd.DataFrame({n: {"Train PR-AUC": average_precision_score(y_train, r["train"]),
                          "Val PR-AUC": average_precision_score(y_val, r["val"]),
                          "Test PR-AUC": average_precision_score(y_test, r["test"]),
                          "Overfit gap (train - test PR-AUC)": average_precision_score(y_train, r["train"]) - average_precision_score(y_test, r["test"]),
                          "Fit time (s)": r["fit_time"], "Test predict time (s)": r["pred_time"]} for n, r in RES.items()}).T
print("\nOverfitting check and timing")
try:
    from IPython.display import display; display(extra.style.format("{:.4f}"))
except Exception:
    print(extra.round(4).to_string())

## 8. Graphs

In [ ]:
# (a) Grouped bar chart of the headline metrics (tuned threshold)
keys = ["Accuracy", "Precision", "Recall (TPR)", "F1", "F2 (recall-weighted)", "MCC", "PR-AUC", "ROC-AUC"]
fig, ax = plt.subplots(figsize=(13, 4.8))
w = 0.16; x = np.arange(len(keys))
for i, n in enumerate(RES):
    vals = table_tuned.loc[n, keys].values.astype(float)
    bars = ax.bar(x + (i - 2) * w, vals, w, label=n, color=COLORS[n], hatch="//" if RES[n]["kind"] == "unsupervised" else None)
    for r_, v in zip(bars, vals):
        ax.text(r_.get_x() + r_.get_width() / 2, v + 0.005, f"{v:.2f}", ha="center", fontsize=6.5, rotation=90)
ax.set_xticks(x); ax.set_xticklabels(keys); ax.set_ylim(-0.08, 1.12); ax.axhline(0, color="k", lw=0.8); ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.08))
ax.set_title("Test-set metrics, tuned threshold (hatched = unsupervised)")
plt.tight_layout(); savefig("03_metrics_bar"); plt.show()

# (b) Heatmap of all rate metrics
hm_cols = ["Accuracy", "Balanced Acc", "Precision", "Recall (TPR)", "Specificity (TNR)", "F1", "F2 (recall-weighted)",
           "MCC", "Cohen kappa", "ROC-AUC", "PR-AUC"]
M = table_tuned[hm_cols].values.astype(float)
fig, ax = plt.subplots(figsize=(12, 3.8))
im = ax.imshow(M, cmap="YlGn", vmin=0, vmax=1, aspect="auto"); ax.grid(False)
ax.set_xticks(range(len(hm_cols))); ax.set_xticklabels(hm_cols, rotation=30, ha="right")
ax.set_yticks(range(len(RES))); ax.set_yticklabels(list(RES))
for (i, j), v in np.ndenumerate(M):
    ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8)
ax.set_title("All metrics at a glance (test set, tuned threshold)"); plt.colorbar(im, fraction=0.02)
plt.tight_layout(); savefig("04_metrics_heatmap"); plt.show()

In [ ]:
# (c) ROC and Precision-Recall curves (all models), with the tuned operating point marked
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for n, r in RES.items():
    fpr, tpr, _ = roc_curve(y_test, r["test"])
    ax[0].plot(fpr, tpr, color=COLORS[n], ls="--" if r["kind"] == "unsupervised" else "-",
               label=f"{n} (AUC {roc_auc_score(y_test, r['test']):.3f})")
    p, rc, _ = precision_recall_curve(y_test, r["test"])
    ax[1].plot(rc, p, color=COLORS[n], ls="--" if r["kind"] == "unsupervised" else "-",
               label=f"{n} (AP {average_precision_score(y_test, r['test']):.3f})")
    ax[1].scatter([table_tuned.loc[n, "Recall (TPR)"]], [table_tuned.loc[n, "Precision"]], color=COLORS[n], edgecolor="k", zorder=4, s=60)
ax[0].plot([0, 1], [0, 1], "k:", lw=1); ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate (recall)")
ax[0].set_title("ROC curves"); ax[0].legend(loc="lower right")
ax[1].axhline(y_test.mean(), color="k", ls=":", lw=1); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ax[1].set_title("Precision-Recall curves (dots = tuned threshold)"); ax[1].legend(loc="lower left")
plt.tight_layout(); savefig("05_roc_pr_curves"); plt.show()

# (d) Confusion matrices side by side
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for a, n in zip(axes, RES):
    cm = np.array([[table_tuned.loc[n, "True Negatives"], table_tuned.loc[n, "False Positives"]],
                   [table_tuned.loc[n, "False Negatives"], table_tuned.loc[n, "True Positives"]]], dtype=int)
    a.imshow(cm, cmap="Blues"); a.grid(False)
    for (i, j), v in np.ndenumerate(cm):
        a.text(j, i, v, ha="center", va="center", fontsize=13, color="white" if v > cm.max() / 2 else "black")
    a.set_xticks([0, 1]); a.set_yticks([0, 1]); a.set_xticklabels(["not pulsar", "pulsar"], fontsize=8); a.set_yticklabels(["not pulsar", "pulsar"], fontsize=8)
    a.set_xlabel("Predicted"); a.set_title(n, fontsize=10)
axes[0].set_ylabel("Actual")
plt.suptitle("Confusion matrices (test set, tuned threshold)", y=1.04); plt.tight_layout()
savefig("06_confusion_matrices"); plt.show()

In [ ]:
# (e) Missed pulsars vs false alarms
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
names = list(RES)
for a, col, ttl in [(ax[0], "False Negatives", f"Missed pulsars (of {int(y_test.sum())} real ones)"),
                    (ax[1], "False Positives", f"False alarms (of {int((y_test == 0).sum())} non-pulsars)")]:
    v = table_tuned[col].values.astype(int)
    bars = a.bar(names, v, color=[COLORS[n] for n in names])
    for r_, val_ in zip(bars, v): a.text(r_.get_x() + r_.get_width() / 2, val_, str(val_), ha="center", va="bottom")
    a.set_title(ttl); a.tick_params(axis="x", rotation=25)
plt.tight_layout(); savefig("07_fn_fp"); plt.show()

# (f) Overfitting check: train vs val vs test PR-AUC
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(names)); w = 0.26
for i, (col, c) in enumerate([("Train PR-AUC", "#4c72b0"), ("Val PR-AUC", "#dd8452"), ("Test PR-AUC", "#55a868")]):
    ax.bar(x + (i - 1) * w, extra.loc[names, col].values.astype(float), w, label=col, color=c)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15); ax.set_ylim(0, 1.05); ax.legend(loc="lower right")
ax.set_title("Overfitting check: a big train-vs-test gap means overfitting")
plt.tight_layout(); savefig("08_overfitting_check"); plt.show()

# (g) Training and prediction time
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(names, extra.loc[names, "Fit time (s)"].astype(float), color=[COLORS[n] for n in names]); ax[0].set_title("Fit time incl. tuning (s)")
ax[1].bar(names, extra.loc[names, "Test predict time (s)"].astype(float), color=[COLORS[n] for n in names]); ax[1].set_title("Test prediction time (s)")
for a in ax: a.tick_params(axis="x", rotation=25); a.set_yscale("log")
plt.tight_layout(); savefig("09_timing"); plt.show()

# (h) Feature importance (tree models)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
pd.Series(xgb.feature_importances_, index=FEATURES).sort_values().plot.barh(ax=ax[0], color=COLORS["XGBoost"], title="XGBoost feature importance")
pd.Series(rf.feature_importances_, index=FEATURES).sort_values().plot.barh(ax=ax[1], color=COLORS["Random Forest"], title="Random Forest feature importance")
plt.tight_layout(); savefig("10_feature_importance"); plt.show()

## 9. Supervised vs unsupervised: what the data looks like
2D PCA of the test set (PCA fit on train). Compare how the labels are laid out against what each unsupervised model finds, and what the best supervised model finds.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(Xtr_s)
Z = pca.transform(Xte_s)
best_sup = table_tuned.loc[SUPERVISED, "F2 (recall-weighted)"].astype(float).idxmax()
panels = [("True labels", y_test),
          ("K-means (k=2) prediction", (RES["K-means (k=2)"]["test"] >= RES["K-means (k=2)"]["thr"]).astype(int)),
          ("Isolation Forest prediction", (RES["Isolation Forest"]["test"] >= RES["Isolation Forest"]["thr"]).astype(int)),
          (f"{best_sup} prediction", (RES[best_sup]["test"] >= RES[best_sup]["thr"]).astype(int))]
fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharex=True, sharey=True)
for a, (ttl, lab) in zip(axes, panels):
    a.scatter(Z[lab == 0, 0], Z[lab == 0, 1], s=4, c="#9aa5b1", alpha=.4, label="not pulsar")
    a.scatter(Z[lab == 1, 0], Z[lab == 1, 1], s=8, c="#d62728", alpha=.8, label="pulsar")
    a.set_title(ttl, fontsize=10); a.set_xlabel("PC1")
axes[0].set_ylabel("PC2"); axes[0].legend(markerscale=3)
plt.tight_layout(); savefig("11_pca_true_vs_predicted"); plt.show()

## 10. Categorizing the pulsars (unsupervised, pulsars only)
The problem statement also asks to *categorize*. Here K-means is run on **only the real pulsars** (all 1,639) to see whether they split into natural groups. We pick k with the silhouette score. HTRU2 has no pulsar sub-type labels, so there is no ground truth to check against, and a low silhouette just means the groups are not well separated.

In [ ]:
X_all = np.vstack([X_train, X_val, X_test]); y_all = np.r_[y_train, y_val, y_test]
P = X_all[y_all == 1]
Ps = StandardScaler().fit_transform(P)
ks = list(range(2, 8)); sil, inertia = [], []
for k in ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Ps)
    sil.append(silhouette_score(Ps, km_k.labels_)); inertia.append(km_k.inertia_)
best_k = ks[int(np.argmax(sil))]
km_p = KMeans(n_clusters=best_k, n_init=10, random_state=SEED).fit(Ps)
pca_p = PCA(n_components=2).fit(Ps); Zp = pca_p.transform(Ps)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(ks, inertia, "o-"); ax[0].set_title("Elbow (inertia)"); ax[0].set_xlabel("k")
ax[1].plot(ks, sil, "o-", color="#d62728"); ax[1].axvline(best_k, ls=":", color="k"); ax[1].set_title(f"Silhouette (best k = {best_k}, {max(sil):.2f})"); ax[1].set_xlabel("k")
sc_ = ax[2].scatter(Zp[:, 0], Zp[:, 1], c=km_p.labels_, cmap="tab10", s=10)
ax[2].set_title(f"Pulsar clusters in PCA space ({pca_p.explained_variance_ratio_.sum():.0%} variance)"); ax[2].set_xlabel("PC1"); ax[2].set_ylabel("PC2")
plt.tight_layout(); savefig("12_pulsar_clusters"); plt.show()

prof = pd.DataFrame(P, columns=FEATURES).groupby(km_p.labels_).mean()
prof.insert(0, "size", pd.Series(km_p.labels_).value_counts().sort_index().values)
prof.index.name = "cluster"
print("Cluster profiles (feature means per cluster):")
try:
    from IPython.display import display; display(prof.round(2))
except Exception:
    print(prof.round(2).to_string())

## 11. Final verdict (auto-generated from the numbers above)

In [ ]:
rank_cols = ["Recall (TPR)", "Precision", "F1", "MCC", "PR-AUC"]
rank_df = table_tuned[rank_cols].astype(float).rank(ascending=False)
overall = rank_df.mean(axis=1).sort_values()
print("Best model per metric (tuned threshold, test set):")
for c in ["Accuracy", "Precision", "Recall (TPR)", "F1", "F2 (recall-weighted)", "MCC", "PR-AUC", "ROC-AUC"]:
    s = table_tuned[c].astype(float); print(f"  {c:22s} {s.idxmax():18s} {s.max():.4f}")
print("\nOverall ranking (mean rank over recall, precision, F1, MCC, PR-AUC; 1 = best):")
for n, v in overall.items(): print(f"  {n:18s} {v:.2f}   [{RES[n]['kind']}]")

sup_best = overall[[n for n in overall.index if RES[n]["kind"] == "supervised"]].index[0]
uns_best = overall[[n for n in overall.index if RES[n]["kind"] == "unsupervised"]].index[0]
f = lambda n, c: float(table_tuned.loc[n, c])
print(f"\nBest supervised: {sup_best} -> recall {f(sup_best,'Recall (TPR)'):.3f}, precision {f(sup_best,'Precision'):.3f}, F1 {f(sup_best,'F1'):.3f}, PR-AUC {f(sup_best,'PR-AUC'):.3f}")
print(f"Best unsupervised: {uns_best} -> recall {f(uns_best,'Recall (TPR)'):.3f}, precision {f(uns_best,'Precision'):.3f}, F1 {f(uns_best,'F1'):.3f}, PR-AUC {f(uns_best,'PR-AUC'):.3f}")
print(f"Gap in F1 (supervised - unsupervised): {f(sup_best,'F1') - f(uns_best,'F1'):+.3f}")
print("Missed pulsars:", {n: int(table_tuned.loc[n, 'False Negatives']) for n in RES})
print("\nTalking points: supervised models use the labels, so they usually win clearly; unsupervised models only find structure,"
      "\nso they are weaker at this task, which is exactly why labels matter. Quote the numbers above in your write-up.")

# save tables + figures
table_tuned.round(4).to_csv("results_tuned_threshold.csv"); table_default.round(4).to_csv("results_default_threshold.csv")
extra.round(4).to_csv("results_overfit_timing.csv")
shutil.make_archive("pulsar_results", "zip", ".", "figures")
try:
    from google.colab import files
    for fn in ["results_tuned_threshold.csv", "results_default_threshold.csv", "results_overfit_timing.csv", "pulsar_results.zip"]:
        files.download(fn)
except ImportError:
    print("\nSaved CSVs and pulsar_results.zip (figures)")